## 0. Kernel setup (run in a terminal, not in this notebook)

Before launching this notebook, create and select a conda environment kernel (`2ndWorkshop`).

### Purdue Gilbreth cluster

```bash
module load conda
conda-env-mod create -n ENV_NAME_HERE -j
module use $HOME/privatemodules
module load conda-env/ENV_NAME_HERE-py3.10.11
```

Replace `ENV_NAME_HERE` with your environment name (`2ndWorkshop`), then select the matching kernel in Jupyter before running the cells below.

Check the env and kernel were created:
```bash
conda env list
jupyter kernelspec list
```

### Local machine (conda)

```bash
conda create -n 2ndWorkshop python=3.10 pip -y
conda activate 2ndWorkshop
pip install ipykernel
python -m ipykernel install --user --name 2ndWorkshop --display-name "Python (2ndWorkshop)"
```

Select the `Python (2ndWorkshop)` kernel, run the install cell below once, then restart the kernel.

**Note:** This notebook only starts the MCP tool server — it doesn't need an LLM backend
at all. Keep its kernel running, then open **`Workshop2_Part2b_Agent.ipynb`** in a
*separate* kernel to connect the agent.


# Workshop 2, Part 2a: MCP Tool Server

This is the **server half** of Part 2. It wraps the same course knowledge base and
academic calendar from Part 1 in three tools — search, calendar lookup, and a
notification writer — and exposes them over HTTP using **MCP** (Model Context Protocol),
so any MCP-compatible agent can discover and call them.

**What you'll do:**
- Define the MCP tools with FastMCP
- Start the server — the last cell blocks and keeps it running

Run this notebook first and leave its kernel running. Then open
**`Workshop2_Part2b_Agent.ipynb`** in a separate kernel — that's where the LangGraph
agent connects to these tools and does the reasoning.

Two notebooks, two kernels, two real separate processes — this is how MCP servers are
used in practice (the server could just as easily be running on a different machine).


## 0. Install dependencies

Run once, then restart the kernel.

In [ ]:
!pip install sentence-transformers faiss-cpu fastmcp python-dotenv


---
# Part B: MCP Server — Exposing Tools over HTTP

**MCP (Model Context Protocol)** is a standard for exposing tools that LLMs can call.
This notebook runs the server in the foreground — the last cell (B5) blocks and keeps it
running, the same way a server process keeps running in a terminal.

**Transport options:**

| Transport | How it works | Best for |
|---|---|---|
| `http` (Streamable HTTP) | Agent connects via HTTP; server runs independently | Production, multi-client, debuggable |
| `stdio` | Client spawns server as a subprocess | CLI tools, local single-client use |


## B1. Imports for the MCP server

In [ ]:
import os
import re
import json
from pathlib import Path

import faiss
from sentence_transformers import SentenceTransformer
from fastmcp import FastMCP

BASE_DIR = Path(".").resolve()
DATA_DIR = BASE_DIR / "boilermaker_ta_data"

print("Data directory:", DATA_DIR)
print("Files:", list(DATA_DIR.iterdir()) if DATA_DIR.exists() else "NOT FOUND")


## B2. SimpleRetriever — shared retrieval helper for MCP tools

In [ ]:
class SimpleRetriever:
    def __init__(self, texts, metadatas, model_name="sentence-transformers/all-MiniLM-L6-v2"):
        self.embedder  = SentenceTransformer(model_name)
        self.texts     = texts
        self.metadatas = metadatas
        self.index     = self._build_index(texts)

    def _build_index(self, texts):
        embeddings = self.embedder.encode(texts, convert_to_numpy=True, show_progress_bar=False)
        if embeddings.ndim == 1:
            embeddings = embeddings.reshape(1, -1)
        faiss.normalize_L2(embeddings)
        index = faiss.IndexFlatIP(embeddings.shape[1])
        index.add(embeddings)
        return index

    def similarity_search(self, query: str, k: int = 3):
        q_emb = self.embedder.encode(query, convert_to_numpy=True)
        if q_emb.ndim == 1:
            q_emb = q_emb.reshape(1, -1)
        faiss.normalize_L2(q_emb)
        scores, ids = self.index.search(q_emb, min(k, len(self.texts)))
        return [
            type("Doc", (), {"page_content": self.texts[idx], "metadata": self.metadatas[idx]})
            for idx in ids[0]
        ]

## B3. Retriever builder helpers

In [ ]:
ANNOUNCEMENTS_FILE = BASE_DIR / "workshop_outputs/announcements.txt"


def _load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _build_retriever():
    """Build a FAISS retriever from the knowledge base.
    For production, cache the vectorstore to avoid rebuilding on every call.
    """
    kb        = _load_json(DATA_DIR / "knowledge_base.json")
    texts     = [doc["text"] for doc in kb]
    metadatas = [{"title": doc.get("title")} for doc in kb]
    return SimpleRetriever(texts, metadatas)


def _build_calendar_retriever():
    """Build a FAISS retriever from academic calendar events.
    Calendar events are flattened to 'date title notes' strings for embedding.
    """
    calendar = _load_json(DATA_DIR / "purdue_calendar.json")
    events   = calendar.get("events", [])
    if not events:
        return None
    texts     = [f"{e['date']} {e['title']} {e['notes']}" for e in events]
    metadatas = [{"date": e.get("date"), "title": e.get("title")} for e in events]
    return SimpleRetriever(texts, metadatas)

## B4. Define the MCP app and its tools

Each `@app.tool` function becomes a tool the LLM agent can call by name.

In [ ]:
app = FastMCP(
    name="boilermaker-ta",
    instructions=(
        "Provides tools for finding course facts, querying the academic calendar, "
        "and writing student notification announcements."
    ),
)


@app.tool
def search_knowledge_base(query: str) -> str:
    """Search a small local knowledge base of course and schedule facts."""
    kb          = _load_json(DATA_DIR / "knowledge_base.json")
    query_terms = re.findall(r"\w+", query.lower())
    scored = []
    for doc in kb:
        text  = doc["text"].lower()
        score = sum(text.count(term) for term in query_terms)
        if score > 0:
            scored.append((score, doc))
    if not scored:
        return "No relevant knowledge found. Try a different question."
    scored.sort(key=lambda item: item[0], reverse=True)
    return "\n\n".join(f"{doc['title']}:\n{doc['text']}" for _, doc in scored[:2])


@app.tool
def retrieve_docs(query: str, k: int = 3) -> str:
    """Retrieve top-k documents from the knowledge base using embeddings + FAISS."""
    try:
        r = _build_retriever()
    except Exception as e:
        return f"Failed to build retriever: {e}"
    docs  = r.similarity_search(query, k=k)
    lines = []
    for d in docs:
        title   = d.metadata.get("title") if d.metadata else "(no title)"
        excerpt = (d.page_content[:400] + "...") if len(d.page_content) > 400 else d.page_content
        lines.append(f"{title}: {excerpt}")
    return "\n\n".join(lines)


@app.tool
def get_academic_calendar(query: str = "next 30 days") -> str:
    """Retrieve calendar events using semantic search (embedding + FAISS)."""
    try:
        r = _build_calendar_retriever()
    except Exception as e:
        return f"Failed to build calendar retriever: {e}"
    if r is None:
        return "Academic calendar is empty."
    docs  = r.similarity_search(query, k=5)
    lines = [
        f"{d.metadata.get('date')} - {d.metadata.get('title')}"
        for d in docs
    ]
    return "Academic calendar events:\n" + "\n".join(lines)


@app.tool
def create_notification(subject: str, body: str) -> str:
    """Write a notification to announcements.txt and return its location."""
    ANNOUNCEMENTS_FILE.parent.mkdir(parents=True, exist_ok=True)
    entry = f"Subject: {subject}\n{body}\n---\n"
    with open(ANNOUNCEMENTS_FILE, "a", encoding="utf-8") as f:
        f.write(entry)
    return f"Notification written to {ANNOUNCEMENTS_FILE}."


print("MCP app defined with tools:", ["search_knowledge_base", "retrieve_docs", "get_academic_calendar", "create_notification"])

## B5. Start the MCP server (this cell blocks)

This cell runs the server in the foreground: it keeps executing — blocking this kernel —
until you stop it. That's expected.

**Leave this cell running**, then open `Workshop2_Part2b_Agent.ipynb` in a separate
kernel to connect as a client.

To stop the server: **Kernel → Interrupt** (or restart the kernel).


In [ ]:
host = os.environ.get("MCP_HOST", "127.0.0.1")
port = int(os.environ.get("MCP_PORT", "8001"))

print(f"Starting MCP server at http://{host}:{port}/mcp")
print("This cell will keep running -- use Kernel > Interrupt to stop the server.")

# Jupyter's kernel already runs an asyncio event loop, so app.run() (which calls
# anyio.run() to start a new one) raises "Already running asyncio in this thread".
# Use the async variant with top-level await instead.
await app.run_async(transport="http", host=host, port=port)


---
## Next: Workshop2_Part2b_Agent.ipynb

With this server running, open **`Workshop2_Part2b_Agent.ipynb`** (a separate kernel) to
connect the LangGraph agent and run the Boilermaker TA.
